# Lab 24. Multivariate, Panel, and Hierarchical Time Series

**Course:** Modern Time Series Analysis and Sequence Data  
**Book repo:** `Book-MachineLearning2sda`  
**Open in Colab:** <https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-24-multivariate-panel-hierarchical-lab.ipynb>

## Independent-study goals

By the end of this lab, you should be able to:

1. Explain why many practical forecasting problems involve several related time series rather than one isolated series.
2. Simulate and organize a small multivariate panel of related series.
3. Compare univariate lag forecasting with multivariate lag forecasting.
4. Build a pooled panel forecasting model using lag, rolling, calendar, and group features.
5. Represent a hierarchy using an aggregation matrix.
6. Diagnose incoherent forecasts and reconcile them by bottom-up and least-squares reconciliation.
7. Connect forecasting accuracy with structural constraints such as
   $$
   \text{total} = \sum_{i=1}^{m} \text{bottom series}_i.
   $$

This lab is written for independent study. Read each explanation before running the code. After each major code block, answer the interpretation questions in your own words.

## 1. Conceptual background

Classical time series methods often begin with one process,
$$
Y_1, Y_2, \ldots, Y_T.
$$
In applications, however, we often observe a collection of related processes:
$$
Y_t = (Y_{1,t},Y_{2,t},\ldots,Y_{d,t})^T.
$$
Examples include:

- sales by region and product,
- hospital visits by department,
- electricity demand by zone,
- economic indicators across countries,
- sensor measurements from several machines.

There are three common structures.

**Multivariate time series.** A vector is observed at each time point. Lagged values of one component may help forecast another component.

**Panel time series.** Many related units are observed repeatedly over time. A pooled model can share information across units.

**Hierarchical time series.** Series are connected by aggregation rules. For example, total sales equals the sum of regional sales, and each regional total equals the sum of its product-level sales.

In this lab we will use one synthetic example throughout: monthly demand for two regions and two products.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(7243)

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

print("Libraries loaded.")

## 2. Simulate a small hierarchical panel

We create four bottom-level series:

- North, Product A
- North, Product B
- South, Product A
- South, Product B

These four bottom-level series can be aggregated into region totals, product totals, and a grand total.

The simulation includes:

- a common trend,
- common annual seasonality,
- region-specific effects,
- product-specific effects,
- correlated shocks,
- autoregressive noise.

The goal is not to imitate any particular dataset. The goal is to create a controlled example where dependence and aggregation are visible.

In [ ]:
T = 240
period = 12
dates = pd.date_range("2005-01-01", periods=T, freq="MS")
t = np.arange(T)

bottom_labels = ["North_A", "North_B", "South_A", "South_B"]
regions = {"North_A": "North", "North_B": "North", "South_A": "South", "South_B": "South"}
products = {"North_A": "A", "North_B": "B", "South_A": "A", "South_B": "B"}

trend = 0.04 * t
annual = 4.0 * np.sin(2 * np.pi * t / period) + 1.5 * np.cos(2 * np.pi * t / period)
slow_factor = np.cumsum(rng.normal(0, 0.08, size=T))

# Correlated innovations across the four bottom series.
correlation = np.array([
    [1.00, 0.55, 0.35, 0.25],
    [0.55, 1.00, 0.20, 0.40],
    [0.35, 0.20, 1.00, 0.50],
    [0.25, 0.40, 0.50, 1.00]
])
std = np.array([1.8, 1.5, 1.6, 1.7])
cov = np.outer(std, std) * correlation
innov = rng.multivariate_normal(mean=np.zeros(4), cov=cov, size=T)

noise = np.zeros((T, 4))
phi = np.array([0.55, 0.40, 0.50, 0.45])
for i in range(4):
    for j in range(1, T):
        noise[j, i] = phi[i] * noise[j - 1, i] + innov[j, i]

base = np.array([34, 28, 31, 25])
region_effect = np.column_stack([
    1.5 * np.sin(2 * np.pi * t / 48),
    1.5 * np.sin(2 * np.pi * t / 48),
    -1.2 * np.cos(2 * np.pi * t / 54),
    -1.2 * np.cos(2 * np.pi * t / 54)
])
product_effect = np.column_stack([
    2.0 * np.cos(2 * np.pi * t / 18),
    -1.0 * np.sin(2 * np.pi * t / 24),
    2.0 * np.cos(2 * np.pi * t / 18),
    -1.0 * np.sin(2 * np.pi * t / 24)
])

Y_bottom = (
    base
    + trend[:, None]
    + annual[:, None]
    + slow_factor[:, None]
    + region_effect
    + product_effect
    + noise
)
Y_bottom = np.maximum(Y_bottom, 1.0)

bottom_df = pd.DataFrame(Y_bottom, index=dates, columns=bottom_labels)
bottom_df.head()

In [ ]:
ax = bottom_df.plot(title="Four bottom-level monthly demand series")
ax.set_xlabel("Date")
ax.set_ylabel("Demand")
plt.show()

total = bottom_df.sum(axis=1)
ax = total.plot(title="Grand total demand")
ax.set_xlabel("Date")
ax.set_ylabel("Demand")
plt.show()

### Interpretation checkpoint 1

1. Which bottom-level series seem most strongly related visually?
2. Do all four series share the same seasonality?
3. Why might forecasting these series separately waste information?
4. Why might forecasting only the total hide useful information?

## 3. Cross-correlation and lagged dependence

For multivariate forecasting, we care about whether one series helps predict another. A simple diagnostic is the lagged correlation.

For two series $x_t$ and $y_t$, we can compute the correlation between $x_t$ and $y_{t-k}$. If this value is large for a positive lag $k$, then past values of $y$ may help predict current values of $x$.

In [ ]:
def lagged_correlation(x, y, max_lag=12):
    """Correlation between x_t and y_{t-lag} for lag = 0, ..., max_lag."""
    x = np.asarray(x)
    y = np.asarray(y)
    rows = []
    for lag in range(max_lag + 1):
        if lag == 0:
            x_part = x
            y_part = y
        else:
            x_part = x[lag:]
            y_part = y[:-lag]
        corr = np.corrcoef(x_part, y_part)[0, 1]
        rows.append({"lag": lag, "corr_x_t_y_t_minus_lag": corr})
    return pd.DataFrame(rows)

corr_table = lagged_correlation(bottom_df["North_A"], bottom_df["South_A"], max_lag=12)
corr_table

In [ ]:
ax = corr_table.plot(x="lag", y="corr_x_t_y_t_minus_lag", marker="o", legend=False)
ax.set_title("Lagged correlation: North_A at time t vs South_A at time t - lag")
ax.set_xlabel("Lag")
ax.set_ylabel("Correlation")
plt.show()

print("Contemporaneous correlation matrix:")
display(bottom_df.corr().round(2))

### Interpretation checkpoint 2

1. Which contemporaneous correlations are largest?
2. What could create correlation among bottom-level demand series?
3. Why is correlation not the same thing as causation?
4. In forecasting, why can correlation still be useful even without a causal interpretation?

## 4. Multivariate lag forecasting

A univariate lag model for a single component might use only its own history:
$$
Y_{i,t} \approx b_i + a_{i,1}Y_{i,t-1} + \cdots + a_{i,p}Y_{i,t-p}.
$$

A multivariate lag model uses the histories of all components:
$$
Y_t \approx b + A_1Y_{t-1} + A_2Y_{t-2} + \cdots + A_pY_{t-p}.
$$

This resembles a vector autoregression, but we will fit it with ridge regression to keep the implementation simple and stable.

In [ ]:
def make_multivariate_lag_matrix(Y, lags):
    """Return X, target, and target time indices for a multivariate lag model."""
    X_rows = []
    targets = []
    times = []
    for current_time in range(lags, len(Y)):
        features = []
        for lag in range(1, lags + 1):
            features.extend(Y[current_time - lag])
        X_rows.append(features)
        targets.append(Y[current_time])
        times.append(current_time)
    return np.asarray(X_rows), np.asarray(targets), np.asarray(times)

lags = 12
train_end = 190

Y = bottom_df.to_numpy()
X_all, y_all, time_index = make_multivariate_lag_matrix(Y, lags=lags)

train_mask = time_index < train_end
test_mask = time_index >= train_end

X_train, X_test = X_all[train_mask], X_all[test_mask]
y_train, y_test = y_all[train_mask], y_all[test_mask]
test_times = time_index[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

multi_model = Ridge(alpha=25.0)
multi_model.fit(X_train_scaled, y_train)
y_pred_multi = multi_model.predict(X_test_scaled)

# Baselines: one-step persistence and seasonal naive.
y_pred_persist = Y[test_times - 1]
y_pred_seasonal = Y[test_times - period]

# Univariate ridge: each output uses only its own lags.
y_pred_uni = np.zeros_like(y_test)
for j in range(Y.shape[1]):
    Xj = []
    yj = []
    tj = []
    for current_time in range(lags, len(Y)):
        Xj.append([Y[current_time - lag, j] for lag in range(1, lags + 1)])
        yj.append(Y[current_time, j])
        tj.append(current_time)
    Xj = np.asarray(Xj)
    yj = np.asarray(yj)
    tj = np.asarray(tj)
    train_j = tj < train_end
    test_j = tj >= train_end
    scaler_j = StandardScaler()
    Xj_train = scaler_j.fit_transform(Xj[train_j])
    Xj_test = scaler_j.transform(Xj[test_j])
    model_j = Ridge(alpha=15.0)
    model_j.fit(Xj_train, yj[train_j])
    y_pred_uni[:, j] = model_j.predict(Xj_test)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

summary = pd.DataFrame({
    "method": ["persistence", "seasonal naive", "univariate ridge", "multivariate ridge"],
    "overall_RMSE": [
        rmse(y_test, y_pred_persist),
        rmse(y_test, y_pred_seasonal),
        rmse(y_test, y_pred_uni),
        rmse(y_test, y_pred_multi)
    ],
    "overall_MAE": [
        mean_absolute_error(y_test, y_pred_persist),
        mean_absolute_error(y_test, y_pred_seasonal),
        mean_absolute_error(y_test, y_pred_uni),
        mean_absolute_error(y_test, y_pred_multi)
    ]
})
summary.round(3)

In [ ]:
per_series = []
methods = {
    "persistence": y_pred_persist,
    "seasonal naive": y_pred_seasonal,
    "univariate ridge": y_pred_uni,
    "multivariate ridge": y_pred_multi
}
for name, pred in methods.items():
    for j, label in enumerate(bottom_labels):
        per_series.append({
            "method": name,
            "series": label,
            "RMSE": rmse(y_test[:, j], pred[:, j]),
            "MAE": mean_absolute_error(y_test[:, j], pred[:, j])
        })

per_series_df = pd.DataFrame(per_series)
display(per_series_df.pivot(index="series", columns="method", values="RMSE").round(3))

j = bottom_labels.index("North_A")
plot_dates = dates[test_times]
plt.plot(plot_dates, y_test[:, j], label="actual")
plt.plot(plot_dates, y_pred_seasonal[:, j], label="seasonal naive")
plt.plot(plot_dates, y_pred_multi[:, j], label="multivariate ridge")
plt.title("Forecast comparison for North_A")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.show()

### Interpretation checkpoint 3

1. Did the multivariate model improve over the univariate model?
2. For which series did multivariate information help most?
3. Why might ridge regularization help when many lagged predictors are used?
4. In a real dataset, what danger would appear if we selected lags using the test set?

## 5. Panel forecasting with pooled features

A panel model uses one long table with columns such as:

- unit or series ID,
- time,
- lag features,
- rolling features,
- seasonal features,
- target value.

Instead of fitting one model per series, a pooled model learns from all rows at once. This can help when individual series are short or noisy.

Here we build a row for each pair `(time, series)` and fit a single ridge regression model.

In [ ]:
# Convert the wide bottom-level data to long panel format.
panel_df = bottom_df.reset_index(names="date").melt(
    id_vars="date", var_name="series", value_name="y"
)
panel_df["t"] = panel_df.groupby("series").cumcount()
panel_df["region"] = panel_df["series"].map(regions)
panel_df["product"] = panel_df["series"].map(products)

# Lag and rolling features must use only the past. The shift(1) is important.
panel_df = panel_df.sort_values(["series", "date"]).copy()
for lag in [1, 2, 3, 12]:
    panel_df[f"lag_{lag}"] = panel_df.groupby("series")["y"].shift(lag)

panel_df["roll3_mean"] = (
    panel_df.groupby("series")["y"]
    .shift(1)
    .rolling(3)
    .mean()
    .reset_index(level=0, drop=True)
)
panel_df["roll6_mean"] = (
    panel_df.groupby("series")["y"]
    .shift(1)
    .rolling(6)
    .mean()
    .reset_index(level=0, drop=True)
)

panel_df["sin12"] = np.sin(2 * np.pi * panel_df["t"] / 12)
panel_df["cos12"] = np.cos(2 * np.pi * panel_df["t"] / 12)

panel_model_df = panel_df.dropna().reset_index(drop=True)
panel_model_df.head()

In [ ]:
feature_numeric = ["t", "lag_1", "lag_2", "lag_3", "lag_12", "roll3_mean", "roll6_mean", "sin12", "cos12"]
feature_categorical = ["series", "region", "product"]

X_panel = pd.concat(
    [
        panel_model_df[feature_numeric],
        pd.get_dummies(panel_model_df[feature_categorical], drop_first=False, dtype=float)
    ],
    axis=1
)
y_panel = panel_model_df["y"].to_numpy()

time_panel = panel_model_df["t"].to_numpy()
train_panel = time_panel < train_end
test_panel = time_panel >= train_end

panel_scaler = StandardScaler()
X_panel_train = panel_scaler.fit_transform(X_panel.loc[train_panel])
X_panel_test = panel_scaler.transform(X_panel.loc[test_panel])

pooled_model = Ridge(alpha=30.0)
pooled_model.fit(X_panel_train, y_panel[train_panel])
y_panel_pred = pooled_model.predict(X_panel_test)

# Compare against seasonal naive at the panel row level.
y_panel_actual = y_panel[test_panel]
y_panel_seasonal = panel_model_df.loc[test_panel, "lag_12"].to_numpy()

eval_panel = pd.DataFrame({
    "method": ["seasonal naive", "pooled panel ridge"],
    "RMSE": [rmse(y_panel_actual, y_panel_seasonal), rmse(y_panel_actual, y_panel_pred)],
    "MAE": [mean_absolute_error(y_panel_actual, y_panel_seasonal), mean_absolute_error(y_panel_actual, y_panel_pred)]
})
eval_panel.round(3)

In [ ]:
# Panel evaluation by series.
panel_test = panel_model_df.loc[test_panel, ["date", "series", "y", "lag_12"]].copy()
panel_test["pooled_pred"] = y_panel_pred

rows = []
for label, group in panel_test.groupby("series"):
    rows.append({
        "series": label,
        "seasonal_naive_RMSE": rmse(group["y"], group["lag_12"]),
        "pooled_panel_RMSE": rmse(group["y"], group["pooled_pred"]),
        "pooled_panel_MAE": mean_absolute_error(group["y"], group["pooled_pred"])
    })

pd.DataFrame(rows).round(3)

### Interpretation checkpoint 4

1. What information is shared across series by the pooled panel model?
2. What information remains series-specific?
3. Why must rolling features use `shift(1)` before computing the rolling mean?
4. When might a pooled model perform worse than separate models?

## 6. Hierarchical time series and aggregation matrices

Let $b_t$ be the vector of bottom-level series. In this example,
$$
 b_t = (\text{North A}, \text{North B}, \text{South A}, \text{South B})^T.
$$

All aggregate series can be written as
$$
 y_t = S b_t,
$$
where $S$ is an aggregation matrix.

For example, the row for the grand total is
$$
(1,1,1,1),
$$
because total demand is the sum of all four bottom-level series.

The key question in hierarchical forecasting is this:

> If we produce separate forecasts for the total, region totals, product totals, and bottom series, will those forecasts add up correctly?

Usually they do not. Reconciliation adjusts forecasts so they satisfy the hierarchy.

In [ ]:
all_labels = [
    "Total",
    "North", "South",
    "Product_A", "Product_B",
    "North_A", "North_B", "South_A", "South_B"
]

S = np.array([
    [1, 1, 1, 1],  # Total
    [1, 1, 0, 0],  # North
    [0, 0, 1, 1],  # South
    [1, 0, 1, 0],  # Product A
    [0, 1, 0, 1],  # Product B
    [1, 0, 0, 0],  # North A
    [0, 1, 0, 0],  # North B
    [0, 0, 1, 0],  # South A
    [0, 0, 0, 1],  # South B
], dtype=float)

Y_all = Y_bottom @ S.T
all_df = pd.DataFrame(Y_all, index=dates, columns=all_labels)
all_df.head()

In [ ]:
# Verify exact coherence of the observed data.
max_total_error = np.max(np.abs(all_df["Total"] - bottom_df.sum(axis=1)))
max_region_error = np.max(np.abs(all_df["North"] - all_df["North_A"] - all_df["North_B"]))
max_product_error = np.max(np.abs(all_df["Product_A"] - all_df["North_A"] - all_df["South_A"]))

print("Max total aggregation error:", max_total_error)
print("Max North aggregation error:", max_region_error)
print("Max Product_A aggregation error:", max_product_error)

## 7. Independent base forecasts are usually incoherent

We now fit a separate one-step-ahead model to each of the nine series:

- total,
- two region totals,
- two product totals,
- four bottom-level series.

Because the models are fit separately, the resulting forecasts will usually not add up exactly.

In [ ]:
def make_univariate_design(y, lags=12):
    X_rows = []
    targets = []
    times = []
    for current_time in range(lags, len(y)):
        row = [current_time]
        row += [y[current_time - lag] for lag in [1, 2, 3, 12]]
        row += [np.sin(2 * np.pi * current_time / 12), np.cos(2 * np.pi * current_time / 12)]
        X_rows.append(row)
        targets.append(y[current_time])
        times.append(current_time)
    return np.asarray(X_rows), np.asarray(targets), np.asarray(times)

def fit_predict_one_series(y, train_end, alpha=20.0):
    X, target, times = make_univariate_design(y, lags=12)
    train = times < train_end
    test = times >= train_end
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train])
    X_test = scaler.transform(X[test])
    model = Ridge(alpha=alpha)
    model.fit(X_train, target[train])
    pred_test = model.predict(X_test)
    pred_train = model.predict(X_train)
    train_resid = target[train] - pred_train
    return pred_test, target[test], times[test], np.var(train_resid, ddof=1)

base_forecasts = []
actual_all_test = []
resid_vars = []
for label in all_labels:
    pred, actual, times_test, resid_var = fit_predict_one_series(all_df[label].to_numpy(), train_end=train_end)
    base_forecasts.append(pred)
    actual_all_test.append(actual)
    resid_vars.append(max(resid_var, 1e-6))

base_forecasts = np.column_stack(base_forecasts)
actual_all_test = np.column_stack(actual_all_test)
resid_vars = np.asarray(resid_vars)

print("Base forecast array shape:", base_forecasts.shape)
print("Actual array shape:", actual_all_test.shape)
print("Test period starts:", dates[times_test[0]].date())

In [ ]:
def coherence_errors(yhat_all):
    """Return basic aggregation errors for an array with columns ordered as all_labels."""
    yhat = np.asarray(yhat_all)
    total_error = yhat[:, 0] - (yhat[:, 5] + yhat[:, 6] + yhat[:, 7] + yhat[:, 8])
    north_error = yhat[:, 1] - (yhat[:, 5] + yhat[:, 6])
    south_error = yhat[:, 2] - (yhat[:, 7] + yhat[:, 8])
    product_a_error = yhat[:, 3] - (yhat[:, 5] + yhat[:, 7])
    product_b_error = yhat[:, 4] - (yhat[:, 6] + yhat[:, 8])
    return pd.DataFrame({
        "total_error": total_error,
        "north_error": north_error,
        "south_error": south_error,
        "product_a_error": product_a_error,
        "product_b_error": product_b_error
    })

base_coherence = coherence_errors(base_forecasts)
base_coherence.abs().describe().round(3)

### Interpretation checkpoint 5

1. Why are the observed data coherent but the independent forecasts incoherent?
2. Which aggregation error is largest on average?
3. Is incoherence an accuracy problem, a communication problem, or both?
4. In what setting would incoherent forecasts be unacceptable?

## 8. Forecast reconciliation

A coherent forecast must have the form
$$
\widetilde y_t = S\widetilde b_t
$$
for some bottom-level forecast vector $\widetilde b_t$.

We compare three approaches.

**Bottom-up reconciliation.** Forecast the bottom-level series, then aggregate upward.

**Ordinary least-squares reconciliation.** Find the coherent vector closest to the independently produced base forecast:
$$
\widetilde y = S(S^T S)^{-1}S^T\widehat y.
$$

**Weighted least-squares reconciliation.** Use estimated residual variances to trust some base forecasts more than others:
$$
\widetilde y = S(S^T W^{-1}S)^{-1}S^T W^{-1}\widehat y.
$$

Here $W$ is a diagonal matrix of forecast error variances estimated from training residuals.

In [ ]:
# Bottom-up reconciliation: keep bottom forecasts and aggregate upward.
bottom_indices_in_all = [5, 6, 7, 8]
bottom_base = base_forecasts[:, bottom_indices_in_all]
forecast_bottom_up = bottom_base @ S.T

# Ordinary least-squares reconciliation.
P_ols = S @ np.linalg.inv(S.T @ S) @ S.T
forecast_ols = base_forecasts @ P_ols.T

# Weighted least-squares reconciliation.
W_inv = np.diag(1.0 / resid_vars)
P_wls = S @ np.linalg.inv(S.T @ W_inv @ S) @ S.T @ W_inv
forecast_wls = base_forecasts @ P_wls.T

for name, forecast in [
    ("base", base_forecasts),
    ("bottom_up", forecast_bottom_up),
    ("ols_reconciled", forecast_ols),
    ("wls_reconciled", forecast_wls)
]:
    errors = coherence_errors(forecast)
    print(name, "max absolute coherence error:", errors.abs().to_numpy().max().round(8))

In [ ]:
def level_rmse_table(actual, forecasts):
    groups = {
        "top": [0],
        "region": [1, 2],
        "product": [3, 4],
        "bottom": [5, 6, 7, 8],
        "all": list(range(actual.shape[1]))
    }
    rows = []
    for method, pred in forecasts.items():
        for level, cols in groups.items():
            rows.append({
                "method": method,
                "level": level,
                "RMSE": rmse(actual[:, cols], pred[:, cols]),
                "MAE": mean_absolute_error(actual[:, cols], pred[:, cols])
            })
    return pd.DataFrame(rows)

forecast_dict = {
    "base independent": base_forecasts,
    "bottom up": forecast_bottom_up,
    "OLS reconciled": forecast_ols,
    "WLS reconciled": forecast_wls
}

recon_eval = level_rmse_table(actual_all_test, forecast_dict)
display(recon_eval.pivot(index="level", columns="method", values="RMSE").round(3))

In [ ]:
plot_dates = dates[times_test]
plt.plot(plot_dates, actual_all_test[:, 0], label="actual total")
plt.plot(plot_dates, base_forecasts[:, 0], label="base total", alpha=0.8)
plt.plot(plot_dates, forecast_bottom_up[:, 0], label="bottom-up total", alpha=0.8)
plt.plot(plot_dates, forecast_wls[:, 0], label="WLS reconciled total", alpha=0.8)
plt.title("Total-level forecasts before and after reconciliation")
plt.xlabel("Date")
plt.ylabel("Demand")
plt.legend()
plt.show()

# Show a few months of coherent WLS forecasts.
wls_sample = pd.DataFrame(forecast_wls[:5], columns=all_labels, index=plot_dates[:5])
wls_sample.round(2)

### Interpretation checkpoint 6

1. Which reconciliation method performs best at the top level?
2. Which method performs best at the bottom level?
3. Why can bottom-up reconciliation be strong for bottom-level accuracy but weaker at aggregate levels?
4. Why might weighted reconciliation be preferable when some base forecasts are much noisier than others?
5. Why should reconciliation be evaluated at multiple levels rather than only on the total?

## 9. Leakage and structural constraints

Two kinds of mistakes are especially common in multivariate, panel, and hierarchical forecasting.

**Time leakage.** A feature accidentally uses future information. For example, a rolling mean computed without shifting can include the target value itself.

**Aggregation leakage.** A model uses a future aggregate to predict a bottom-level series at the same future time. That aggregate is not known when the forecast is made.

Structural constraints are useful, but they must be applied in a way consistent with the forecasting timeline.

In [ ]:
# Demonstrate a leakage-prone rolling feature.
leak_demo = panel_df[panel_df["series"] == "North_A"].copy()
leak_demo["bad_roll3_includes_today"] = leak_demo["y"].rolling(3).mean()
leak_demo["good_roll3_past_only"] = leak_demo["y"].shift(1).rolling(3).mean()

leak_demo[["date", "y", "bad_roll3_includes_today", "good_roll3_past_only"]].head(8)

### Interpretation checkpoint 7

Look at the first rows of the leakage demonstration.

1. Why does the bad rolling mean include information unavailable at forecast time?
2. Why does `shift(1)` fix the issue?
3. Give one example of aggregation leakage in a sales forecasting problem.

## 10. Exercises

Complete these exercises after running the notebook.

### Exercise 1. Change the lag order

Change `lags = 12` in the multivariate lag model to another value such as 3, 6, 18, or 24. Compare the test RMSE values.

Questions:

1. Does a longer lag history always improve accuracy?
2. What role does regularization play as the number of features grows?

### Exercise 2. Remove seasonal features from the panel model

Refit the pooled panel model after removing `sin12` and `cos12`.

Questions:

1. Does performance get worse?
2. Which series is most affected?

### Exercise 3. Compare hierarchy levels

Using `recon_eval`, identify the best method separately for:

- top level,
- region level,
- product level,
- bottom level.

Question: Why is there no single method that must be best at every level?

### Exercise 4. Try stronger or weaker reconciliation weights

Modify the residual variance vector used in weighted reconciliation. For example, inflate the variance of the total-level base forecast.

Question: How does this change the reconciled forecast?

## 11. Mini-project options

Choose one mini-project for deeper practice.

### Mini-project A. Hierarchical demand forecasting report

Write a short report comparing independent, bottom-up, OLS-reconciled, and WLS-reconciled forecasts. Include:

- one plot,
- one accuracy table,
- one coherence-error table,
- a recommendation for which method should be used.

### Mini-project B. New hierarchy

Change the hierarchy to include three regions or three products. Rebuild the aggregation matrix $S$ and repeat reconciliation.

### Mini-project C. Panel model with exogenous variables

Add a simulated promotion variable or price variable. Make sure it is known at forecast time. Compare a model with and without the exogenous variable.

### Mini-project D. Multivariate lag interpretation

Inspect the coefficient matrix of the multivariate ridge model. Which lagged series appear most important for predicting each target series? Explain carefully, remembering that standardized variables and collinearity affect interpretation.

## 12. AI-assisted study prompts

Use these prompts to check your understanding. Do not ask an AI system to simply solve the lab for you. Ask it to explain, critique, or quiz you.

1. *Explain the difference between multivariate, panel, and hierarchical time series using a sales-by-region-and-product example.*
2. *Why does a rolling mean feature need to be shifted before it is used for forecasting? Give a concrete leakage example.*
3. *Explain the aggregation matrix S in hierarchical forecasting. What does each row represent?*
4. *Compare bottom-up reconciliation and least-squares reconciliation. When might each be preferred?*
5. *Quiz me on forecast coherence, aggregation constraints, and time leakage. Ask one question at a time.*

## 13. Summary

In this lab you learned how to organize and model related time series in three complementary ways:

- A **multivariate model** uses lagged values from several series.
- A **panel model** pools examples across related units.
- A **hierarchical model** respects aggregation constraints.

The most important practical lesson is that forecasting is not only about minimizing error for one series. In many real systems, forecasts must also be consistent with known structure.